# Step 5: LLM Inference — Llama-2-7B-Chat

This is **1 of 4** parallel inference notebooks (Rooein et al., 2024 reproduction).  
Each notebook runs independently in its own Colab tab with a GPU runtime.  

| Notebook | Model |
|----------|-------|
| **This notebook** | **Llama-2-7B-Chat** |
| Notebook 2 | Llama-2-13B-Chat |
| Notebook 3 | Mistral-7B-Instruct-v0.2 |
| Notebook 4 | Gemma-7B-IT |

In [1]:
# bitsandbytes version compatibility:
#   0.42.0 (paper) = too old for Colab's CUDA 12.8
#   0.45+  (latest) = removed sync_gpu, breaks transformers 4.38.0
#   0.44.1 = supports CUDA 12.8 AND compatible with transformers 4.38.0
#!pip install -q bitsandbytes==0.44.1
!pip install -q transformers==4.38.0 accelerate==0.27.0
!pip install -q datasets
!pip install -q -U bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.7/279.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.8 MB/s eta 0:00:00


In [2]:
import os, json, time
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification-2'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
PROMPT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'prompt_metrics')
os.makedirs(PROMPT_DIR, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - need GPU!'}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
Memory: 85.1 GB


In [3]:
# ---- Model Configuration (ONLY CELL THAT DIFFERS BETWEEN NOTEBOOKS) ----
CURRENT_MODEL = 'llama2-7b'
model_id = 'meta-llama/Llama-2-7b-chat-hf'
print(f"Model: {CURRENT_MODEL} ({model_id})")

Model: llama2-7b (meta-llama/Llama-2-7b-chat-hf)


In [4]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
df_all = pd.concat([df_train, df_test], ignore_index=True)
df_all['split'] = ['train'] * len(df_train) + ['test'] * len(df_test)

with open(os.path.join(PROMPT_DIR, 'prompt_questions.json')) as f:
    ALL_PROMPTS = json.load(f)

print(f"Texts: {len(df_all)} | Prompts: {len(ALL_PROMPTS)} | Total calls: {len(df_all)*len(ALL_PROMPTS):,}")

Texts: 4548 | Prompts: 63 | Total calls: 286,524


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

print(f"Loading {CURRENT_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map='auto',
)
model.eval()
print(f"Model loaded: {CURRENT_MODEL}")

Loading llama2-7b...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model loaded: llama2-7b


In [6]:
# Auto-detect batch size based on GPU memory
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_mem_gb > 35:       # A100 (40GB)
        BATCH_SIZE = 16
    elif gpu_mem_gb > 20:     # A10G (24GB)
        BATCH_SIZE = 8
    else:                     # T4 (16GB)
        BATCH_SIZE = 4
else:
    BATCH_SIZE = 1

print(f"GPU memory: {gpu_mem_gb:.1f} GB \u2192 BATCH_SIZE = {BATCH_SIZE}")


def build_prompt(text, question):
    return (
        f"Read the following text and answer the question with only 'yes' or 'no'.\n\n"
        f"Text: {text}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )

def parse_yes_no(response):
    response_lower = response.strip().lower()
    if response_lower.startswith('yes'):
        return 1
    elif response_lower.startswith('no'):
        return 0
    if 'yes' in response_lower:
        return 1
    elif 'no' in response_lower:
        return 0
    return 0


@torch.no_grad()
def _batch_generate(prompts, max_new_tokens=10):
    """Core batch generation \u2014 may raise OOM."""
    tokenizer.padding_side = 'left'
    inputs = tokenizer(
        prompts, return_tensors='pt', truncation=True,
        max_length=2048, padding=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=None, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

    input_length = inputs['input_ids'].shape[1]
    responses = []
    for i in range(len(prompts)):
        new_tokens = outputs[i][input_length:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))

    del inputs, outputs
    torch.cuda.empty_cache()
    return responses


@torch.no_grad()
def get_llm_responses_batch(prompts, max_new_tokens=10):
    """
    Batch inference with automatic OOM fallback.
    Tries full batch first. If OOM, falls back to one-at-a-time for that batch.
    """
    try:
        return _batch_generate(prompts, max_new_tokens)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        responses = []
        for prompt in prompts:
            resp = _batch_generate([prompt], max_new_tokens)
            responses.append(resp[0])
        return responses


# Quick test
test_resp = get_llm_responses_batch([
    build_prompt("The sun is a star.", "Is this text suitable for elementary school?")
])
print(f"Test: '{test_resp[0]}' \u2192 {parse_yes_no(test_resp[0])}")
print(f"Batch inference ready! BATCH_SIZE={BATCH_SIZE}")


GPU memory: 85.1 GB → BATCH_SIZE = 16


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Test: '' → 0
Batch inference ready! BATCH_SIZE=16


In [7]:
progress_path = os.path.join(PROMPT_DIR, f'{CURRENT_MODEL}_progress.csv')
if os.path.exists(progress_path):
    existing = pd.read_csv(progress_path)
    start_idx = len(existing)
    results = existing.to_dict('records')
    print(f"RESUMING from text {start_idx}/{len(df_all)}")
else:
    start_idx = 0
    results = []
    print(f"Starting fresh: 0/{len(df_all)}")
print(f"Remaining: {len(df_all) - start_idx} texts")

RESUMING from text 4548/4548
Remaining: 0 texts


In [8]:
SAVE_EVERY = 50

t_start = time.time()
texts_done = 0
total_remaining = len(df_all) - start_idx

for text_idx in tqdm(range(start_idx, len(df_all)), desc=f'{CURRENT_MODEL} batched'):
    row = df_all.iloc[text_idx]
    text = str(row['full_text'])
    row_results = {'text_idx': text_idx, 'split': row['split']}

    all_prompts_for_text = [build_prompt(text, q) for q in ALL_PROMPTS]

    for batch_start in range(0, len(all_prompts_for_text), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(all_prompts_for_text))
        batch_prompts = all_prompts_for_text[batch_start:batch_end]
        batch_responses = get_llm_responses_batch(batch_prompts, max_new_tokens=10)
        for j, response in enumerate(batch_responses):
            prompt_idx = batch_start + j
            row_results[f'prompt_{prompt_idx}'] = parse_yes_no(response)

    results.append(row_results)
    texts_done += 1

    if (text_idx + 1) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(progress_path, index=False)
        elapsed = time.time() - t_start
        rate = texts_done / elapsed
        remaining = (total_remaining - texts_done) / rate / 60
        print(f"  Checkpoint {text_idx+1}/{len(df_all)} | {rate:.2f} texts/sec | ~{remaining:.0f} min left")

df_prompt_results = pd.DataFrame(results)
df_prompt_results.to_csv(progress_path, index=False)
print(f"\nDone! {len(df_prompt_results)} texts in {(time.time()-t_start)/60:.1f} min")

llama2-7b batched: 0it [00:00, ?it/s]


Done! 4548 texts in 0.0 min


In [9]:
prompt_cols = [c for c in df_prompt_results.columns if c.startswith('prompt_')]

train_mask = df_prompt_results['split'] == 'train'
test_mask = df_prompt_results['split'] == 'test'

train_prompts = df_prompt_results.loc[train_mask, prompt_cols].reset_index(drop=True)
test_prompts = df_prompt_results.loc[test_mask, prompt_cols].reset_index(drop=True)

train_prompts['education_level'] = df_train['education_level'].values
test_prompts['education_level'] = df_test['education_level'].values

train_path = os.path.join(PROMPT_DIR, f'train_prompt_metrics_{CURRENT_MODEL}.csv')
test_path = os.path.join(PROMPT_DIR, f'test_prompt_metrics_{CURRENT_MODEL}.csv')
train_prompts.to_csv(train_path, index=False)
test_prompts.to_csv(test_path, index=False)

print(f"Saved: {train_prompts.shape} train, {test_prompts.shape} test")
print(f"  {train_path}")
print(f"  {test_path}")
print(f"\n{CURRENT_MODEL} COMPLETE!")

Saved: (3638, 64) train, (910, 64) test
  /content/drive/MyDrive/text-difficulty-classification-2/outputs/prompt_metrics/train_prompt_metrics_llama2-7b.csv
  /content/drive/MyDrive/text-difficulty-classification-2/outputs/prompt_metrics/test_prompt_metrics_llama2-7b.csv

llama2-7b COMPLETE!
